# Lab 17-13 · Packets, Routes, Windows and Firewalls

Six parts, each built on `netlab.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import netlab as nl
rng = np.random.default_rng(1713)

## Part 1 · Overhead

*Predict:* the TCP/IPv4 wire efficiency for 64-, 512- and 1460-byte payloads.

In [2]:
for L in (64, 512, 1460):
    print(f"payload {L:5d} B: wire {nl.wire_bytes(L + nl.TCP + nl.IPV4)} B, efficiency {100 * nl.efficiency(L):.1f}%")
L = np.arange(1, 1461)
plt.plot(L, [100 * nl.efficiency(x) for x in L]); plt.xscale("log")
plt.xlabel("TCP payload (bytes)"); plt.ylabel("wire efficiency (%)"); plt.show()
print(f"10 Gb/s port, minimum frames: {nl.max_frame_rate(10e9) / 1e6:.2f} Mpps")

payload    64 B: wire 142 B, efficiency 45.1%
payload   512 B: wire 590 B, efficiency 86.8%
payload  1460 B: wire 1538 B, efficiency 94.9%
10 Gb/s port, minimum frames: 14.88 Mpps


## Part 2 · Addressing

*Predict:* the prefix and number of subnets for 1000-host subnets of 10.20.0.0/16.

In [3]:
p = nl.prefix_for_hosts(1000)
subs = nl.subnet("10.20.0.0/16", p)
print(f"/{p}: {nl.usable_hosts(p)} hosts each, {len(subs)} subnets; first three {[str(s) for s in subs[:3]]}")
table = {"0.0.0.0/0": "ISP", "10.20.0.0/16": "core", "10.20.4.0/22": "building-2", "10.20.5.0/24": "lab"}
for a in ("10.20.5.17", "10.20.6.1", "10.20.200.1", "192.0.2.1"):
    print(f"{a:12s} -> {nl.longest_prefix_match(table, a)}")

/22: 1022 hosts each, 64 subnets; first three ['10.20.0.0/22', '10.20.4.0/22', '10.20.8.0/22']
10.20.5.17   -> lab
10.20.6.1    -> building-2
10.20.200.1  -> core
192.0.2.1    -> ISP


## Part 3 · Routing

*Predict:* the shortest distances from S in the six-router network, and how long a chain counts to infinity.

In [4]:
g = {"S": {"A": 4, "B": 1}, "A": {"S": 4, "B": 2, "C": 5}, "B": {"S": 1, "A": 2, "D": 8},
     "C": {"A": 5, "D": 3, "T": 2}, "D": {"B": 8, "C": 3, "T": 6}, "T": {"C": 2, "D": 6}}
dist, prev = nl.dijkstra(g, "S")
print("Dijkstra:", dist, " path to T:", nl.path(prev, "T"))
print("Bellman-Ford:", nl.bellman_ford(g, "S"))
# distance vector after a failure: a three-router chain A-B-C losing link B-C, then the six-router network losing C-T
chain = {"A": {"B": 1}, "B": {"A": 1, "C": 1}, "C": {"B": 1}}
for sh in (False, True):
    r, d = nl.dv_after_failure(chain, ("B", "C"), "C", split_horizon=sh)
    print(f"chain, split horizon {sh}: {r} rounds to converge, distances {d}")
for sh in (False, True):
    r, d = nl.dv_after_failure(g, ("C", "T"), "T", split_horizon=sh)
    print(f"six routers, C-T fails, split horizon {sh}: {r} rounds, distances {d}")

Dijkstra: {'S': 0, 'A': 3, 'B': 1, 'C': 8, 'D': 9, 'T': 10}  path to T: ['S', 'B', 'A', 'C', 'T']
Bellman-Ford: ({'S': 0, 'A': 3, 'B': 1, 'C': 8, 'D': 9, 'T': 10}, 4)
chain, split horizon False: 15 rounds to converge, distances {'A': 16, 'B': 16, 'C': 0}
chain, split horizon True: 2 rounds to converge, distances {'A': 16, 'B': 16, 'C': 0}
six routers, C-T fails, split horizon False: 8 rounds, distances {'S': 15, 'A': 14, 'B': 14, 'C': 9, 'D': 6, 'T': 0}
six routers, C-T fails, split horizon True: 4 rounds, distances {'S': 15, 'A': 14, 'B': 14, 'C': 9, 'D': 6, 'T': 0}


## Part 4 · TCP

*Predict:* the mean AIMD window at loss rates $10^{-2}$ to $10^{-5}$, and the throughput limits on a 1 Gb/s, 40 ms path.

In [5]:
for p in (1e-2, 1e-3, 1e-4, 1e-5):
    w = nl.simulate_aimd(p, 100_000 if p > 1e-5 else 300_000, rng)
    print(f"p = {p:.0e}: simulated mean window {w:7.1f}, sqrt(3/(2p)) = {math.sqrt(1.5 / p):7.1f}")
print(f"BDP {nl.bdp_bytes(1e9, 0.04) / 1e6:.2f} MB; unscaled window limit {nl.window_throughput(65535, 0.04) / 1e6:.1f} Mb/s; "
      f"shift needed {nl.window_shift_needed(nl.bdp_bytes(1e9, 0.04))}; loss-limited at 1e-5: {nl.mathis_throughput(1460, 0.04, 1e-5) / 1e6:.0f} Mb/s")
for s in nl.rto_sequence([0.1, 0.12, 0.08, 0.2]):
    print("SRTT {:.4f} s, RTTVAR {:.4f} s, RTO {:.4f} s -> used {:.1f} s".format(*s))

p = 1e-02: simulated mean window    12.9, sqrt(3/(2p)) =    12.2
p = 1e-03: simulated mean window    40.8, sqrt(3/(2p)) =    38.7
p = 1e-04: simulated mean window   132.8, sqrt(3/(2p)) =   122.5


p = 1e-05: simulated mean window   410.4, sqrt(3/(2p)) =   387.3
BDP 5.00 MB; unscaled window limit 13.1 Mb/s; shift needed 7; loss-limited at 1e-5: 113 Mb/s
SRTT 0.1000 s, RTTVAR 0.0500 s, RTO 0.3000 s -> used 1.0 s
SRTT 0.1025 s, RTTVAR 0.0425 s, RTO 0.2725 s -> used 1.0 s
SRTT 0.0997 s, RTTVAR 0.0375 s, RTO 0.2497 s -> used 1.0 s
SRTT 0.1122 s, RTTVAR 0.0532 s, RTO 0.3250 s -> used 1.0 s


## Part 5 · Queues

*Predict:* the mean time in an M/M/1 system at 50%, 80% and 95% load, in service times.

In [6]:
for rho in (0.5, 0.8, 0.95):
    print(f"rho = {rho}: simulated {nl.simulate_mm1(rho, 1.0, 200_000, rng):.2f}, formula {nl.mm1_delay(rho, 1.0):.2f}")

rho = 0.5: simulated 2.01, formula 2.00
rho = 0.8: simulated 4.93, formula 5.00
rho = 0.95: simulated 21.15, formula 20.00


## Part 6 · Security

*Predict:* which of five packets the firewall of Problem 17-13.14 allows, and brute-force times for 56-, 80- and 128-bit keys.

In [7]:
rules = [("allow", {"proto": "tcp", "dport": 443}), ("allow", {"proto": "tcp", "dport": 22, "src": "10.0.0.5"}),
         ("deny", {"proto": "tcp", "dport": 22}), ("allow", {"proto": "udp", "dport": 53})]
pkts = [{"proto": "tcp", "dport": 443, "src": "203.0.113.7"}, {"proto": "tcp", "dport": 22, "src": "10.0.0.5"},
        {"proto": "tcp", "dport": 22, "src": "10.0.0.6"}, {"proto": "udp", "dport": 53, "src": "198.51.100.2"},
        {"proto": "tcp", "dport": 80, "src": "10.0.0.5"}]
print([nl.firewall(rules, p) for p in pkts])
for bits in (56, 80, 128):
    print(f"{bits}-bit key at 1e12 guesses/s: {nl.brute_force_years(bits, 1e12):.3g} years")
print(f"64-bit hash, 50% collision after {nl.birthday_trials(64):.3g} hashes")

['allow', 'allow', 'deny', 'allow', 'deny']
56-bit key at 1e12 guesses/s: 0.00114 years
80-bit key at 1e12 guesses/s: 1.92e+04 years
128-bit key at 1e12 guesses/s: 5.39e+18 years
64-bit hash, 50% collision after 5.06e+09 hashes
